In [ ]:
# ==========================================
# CELL 1: PROJECT INSTALLATIONS
# ==========================================
# Run this once on a new machine to install required libraries.

!pip install tensorflow librosa numpy scikit-learn demucs

In [1]:
# ==========================================
# CELL 2: IMPORTS, SETTINGS & AUDIO ENGINE
# ==========================================

import os
import sys
import subprocess
import warnings
import numpy as np
import librosa
import tensorflow as tf
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras import layers, models
from sklearn.model_selection import train_test_split
from tensorflow.keras.utils import to_categorical

warnings.filterwarnings('ignore')


# --- LOCAL FFMPEG BYPASS ---
bin_dir = os.path.join(os.getcwd(), "ffmpeg_local", "ffmpeg-master-latest-win64-gpl-shared", "bin")
env = os.environ.copy()
env["PATH"] = bin_dir + os.pathsep + env["PATH"]

# --- UNIVERSAL SPECTROGRAM FUNCTION ---
def extract_mel_spectrogram(y, sr):
    mel_spec = librosa.feature.melspectrogram(y=y, sr=sr, n_fft=2048, hop_length=512, n_mels=128)
    return librosa.power_to_db(mel_spec, ref=np.max)

print("✅ Libraries loaded, Settings saved, and Audio Engine ready!")

✅ Libraries loaded, Settings saved, and Audio Engine ready!


In [ ]:
# ==========================================
# CELL 3: CNN TRAINING ARCHIVE (DO NOT RUN UNLESS RETRAINING)
# ==========================================

DATA_PATH = r"E:\ml-instrument detection\data\IRMAS-TrainingData"
TARGET_SR = 22050
WINDOW_DURATION = 1.0

print("1. Processing training data into Spectrograms...")
X, y_labels = [], []

for instrument in os.listdir(DATA_PATH):
    folder = os.path.join(DATA_PATH, instrument)
    if not os.path.isdir(folder): continue
        
    for file in os.listdir(folder):
        if not file.endswith(".wav"): continue
        try:
            audio, sr = librosa.load(os.path.join(folder, file), sr=TARGET_SR)
            samples_per_window = int(WINDOW_DURATION * TARGET_SR)
            
            for i in range(0, len(audio), samples_per_window):
                window = audio[i : i + samples_per_window]
                if len(window) < samples_per_window:
                    window = np.pad(window, (0, samples_per_window - len(window)))
                
                spectrogram = extract_mel_spectrogram(window, TARGET_SR)
                if spectrogram.shape[1] == 44: 
                    X.append(spectrogram)
                    y_labels.append(instrument)
        except:
            pass 

# --- PREPARE DATA ---
X = np.array(X)[..., np.newaxis] 
label_encoder = LabelEncoder()
y_categorical = to_categorical(label_encoder.fit_transform(y_labels))
X_train, X_test, y_train, y_test = train_test_split(X, y_categorical, test_size=0.2, random_state=42)

# --- BUILD & TRAIN CNN ---
print("\n2. Training CNN Model...")
model = models.Sequential([
    layers.Conv2D(32, kernel_size=(3, 3), activation='relu', input_shape=X_train.shape[1:]),
    layers.MaxPooling2D(pool_size=(2, 2)),
    layers.BatchNormalization(),
    layers.Conv2D(64, kernel_size=(3, 3), activation='relu'),
    layers.MaxPooling2D(pool_size=(2, 2)),
    layers.BatchNormalization(),
    layers.Conv2D(128, kernel_size=(3, 3), activation='relu'),
    layers.MaxPooling2D(pool_size=(2, 2)),
    layers.BatchNormalization(),
    layers.Flatten(),
    layers.Dense(256, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(len(label_encoder.classes_), activation='sigmoid')
])

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
early_stopping = tf.keras.callbacks.EarlyStopping(monitor='val_accuracy', patience=5, restore_best_weights=True)
model.fit(X_train, y_train, epochs=30, batch_size=32, validation_data=(X_test, y_test), callbacks=[early_stopping])

# --- SAVE ---
model.save('instrument_model.keras')
np.save('label_classes.npy', label_encoder.classes_)
print("\n✅ Training Complete. Model and Labels saved to disk!")

In [3]:
# ==========================================
# CELL 4: AUDIO SEPARATION (DEMUCS)
# ==========================================
# --- CONTROL CENTER ---
AUDIO_FILE = "king.wav"
MODEL_PATH = 'instrument_model.keras'
LABELS_PATH = 'label_classes.npy'

CONFIDENCE_THRESHOLD = 0.10  # AI must be 10% sure to log an instrument
MAX_SILENCE_GAP = 2.0        # Merge timestamps if the silence gap is <= 2 seconds
MIN_PLAY_TIME = 1.5          # Ignore random noises shorter than 1.5 seconds
print(f"🎧 Ripping '{AUDIO_FILE}' into isolated stems... (This takes 1-2 minutes)")

result = subprocess.run(
    [sys.executable, "-m", "demucs", AUDIO_FILE], 
    capture_output=True, text=True, encoding='utf-8', errors='replace', env=env
)

if result.returncode != 0:
    print("🚨 DEMUCS CRASHED. Log:\n", result.stderr)
else:
    print("✅ Separation Complete! Tracks are safely saved in the 'separated' folder.")

🎧 Ripping 'king.wav' into isolated stems... (This takes 1-2 minutes)
✅ Separation Complete! Tracks are safely saved in the 'separated' folder.


In [5]:
# ==========================================
# CELL 5: AI SCANNING & TIMESTAMP GENERATION
# ==========================================
print("🔍 Loading CNN Brain and Scanning isolated stems...")

# --- 1. LOAD MODELS ---
try:
    model = tf.keras.models.load_model(MODEL_PATH)
    label_encoder = LabelEncoder()
    label_encoder.classes_ = np.load(LABELS_PATH, allow_pickle=True)
except Exception as e:
    print(f"🚨 FAILED TO LOAD MODEL. Error: {e}")
    sys.exit()

# --- 2. SCAN THE STEMS ---
stem_folder = os.path.join("separated", "htdemucs", AUDIO_FILE.replace(".wav", ""))
stems = ["vocals.wav", "drums.wav", "bass.wav", "other.wav"]
time_templates = []

for stem in stems:
    stem_path = os.path.join(stem_folder, stem)
    if not os.path.exists(stem_path): continue
        
    audio_data, sample_rate = librosa.load(stem_path, sr=22050)
    window_samples = int(1.0 * sample_rate)
    hop_samples = int(0.5 * sample_rate)    

    for start in range(0, len(audio_data) - window_samples, hop_samples):
        window = audio_data[start : start + window_samples]
        if len(window) < window_samples:
            window = np.pad(window, (0, window_samples - len(window)))
            
        spectrogram = extract_mel_spectrogram(window, sample_rate)
        if spectrogram.shape[1] != 44: continue
        
        probabilities = model.predict(spectrogram[np.newaxis, ..., np.newaxis], verbose=0)[0]
        
        for idx, confidence in enumerate(probabilities):
            if confidence > CONFIDENCE_THRESHOLD:
                time_templates.append({
                    "start": start / sample_rate,
                    "end": (start + window_samples) / sample_rate,
                    "instrument": label_encoder.inverse_transform([idx])[0]
                })
# --- 3. MERGE TIMESTAMPS & PRINT ---
print("\n--- CONTINUOUS INSTRUMENT TIMESTAMPS ---")
instrument_groups = {}
for p in time_templates:
    instrument_groups.setdefault(p['instrument'], []).append((p['start'], p['end']))

for inst, timings in instrument_groups.items():
    timings.sort(key=lambda x: x[0])
    
    continuous_blocks = []
    for start, end in timings:
        if not continuous_blocks:
            continuous_blocks.append([start, end])
        else:
            if start - continuous_blocks[-1][1] <= MAX_SILENCE_GAP:
                continuous_blocks[-1][1] = max(continuous_blocks[-1][1], end)
            else:
                continuous_blocks.append([start, end])
    
    print(f"\n🎵 {inst.upper()} detected during:")
    for b_start, b_end in continuous_blocks:
        duration = b_end - b_start
        if duration >= MIN_PLAY_TIME:
            print(f"  ▶ [{b_start:05.1f}s  -->  {b_end:05.1f}s]  (Played for {duration:.1f}s)")

print("\n✅ Analysis Complete!")

🔍 Loading CNN Brain and Scanning isolated stems...

--- CONTINUOUS INSTRUMENT TIMESTAMPS ---

🎵 VOI detected during:
  ▶ [000.0s  -->  009.5s]  (Played for 9.5s)
  ▶ [012.0s  -->  023.5s]  (Played for 11.5s)
  ▶ [026.0s  -->  078.5s]  (Played for 52.5s)
  ▶ [086.0s  -->  122.0s]  (Played for 36.0s)

🎵 DRU detected during:
  ▶ [000.0s  -->  126.5s]  (Played for 126.5s)

🎵 GAC detected during:
  ▶ [000.0s  -->  022.5s]  (Played for 22.5s)
  ▶ [026.5s  -->  031.0s]  (Played for 4.5s)
  ▶ [047.5s  -->  055.5s]  (Played for 8.0s)
  ▶ [067.0s  -->  069.0s]  (Played for 2.0s)
  ▶ [101.5s  -->  104.5s]  (Played for 3.0s)
  ▶ [121.5s  -->  124.0s]  (Played for 2.5s)

🎵 GEL detected during:
  ▶ [000.5s  -->  008.0s]  (Played for 7.5s)
  ▶ [028.5s  -->  032.5s]  (Played for 4.0s)
  ▶ [036.0s  -->  043.0s]  (Played for 7.0s)
  ▶ [048.0s  -->  062.5s]  (Played for 14.5s)
  ▶ [082.5s  -->  092.0s]  (Played for 9.5s)
  ▶ [095.0s  -->  126.5s]  (Played for 31.5s)

🎵 SAX detected during:
  ▶ [001.0s  -